### 1. Read and prepare the cleaned dataset ###

In [1]:
### Import necessary libraries ###
import pandas as pd
import numpy as np

import helper_func as hf

In [ ]:
data_path = './Cleaned_VLE_Data.csv'
df = pd.read_csv(data_path)
print(df.head())

In [ ]:
df_oh, cat_map = hf.one_hot_encode(df.drop(columns=['Component 1', 'Component 2']), drop_first=True, dummy_na=False, prefix_sep="_")
df_oh['Component 1'] = df['Component 1']
df_oh['Component 2'] = df['Component 2']

In [ ]:
#df_with_smiles = hf.get_smiles(df)
#df_processed = hf.get_descriptors(df_oh, mol_col=['mol1', 'mol2'], save_dir="./processed_data")

#### 2. Train an XGBoost model ###

In [10]:
### Import necessary libraries for modeling ###
from sklearn.model_selection import train_test_split, cross_validate
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
### Define features and target variable ###
data = pd.read_csv('./processed_data/dataset_with_descriptors.csv')
features = [col for col in data.columns if col != 'Mole fraction']
target = 'Mole fraction'
data = data.dropna(subset=features)
X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
y = data[target]

### Split the data into training and testing sets ###
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_hold, y_hold, test_size=0.5, random_state=42)

C:\Users\admin\AppData\Local\Temp\ipykernel_35768\3311501983.py:2: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('./processed_data/dataset_with_descriptors.csv')


In [4]:
### Train the XGBoost model ###
def train_xgboost(X_train, y_train, X_val, y_val):
    X_train, X_val = hf.convert_to_numeric(X_train), hf.convert_to_numeric(X_val)
    model = XGBRegressor(n_jobs = -1, n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    ### Evaluate the model on the test set ###
    y_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    print(f"Test MSE: {mse:.4f}")
    print(f"Test R^2 Score: {r2:.4f}")
    return

In [5]:
X_test.head(5)

,value,"Temperature, K","Pressure, kPa",property_Activity coefficient,"property_Amount density, mol/m3","property_Binary diffusion coefficient, m2/s",property_Compressibility factor,"property_Electrical conductivity, S/m","property_Excess molar Gibbs energy, kJ/mol","property_Excess molar enthalpy (molar enthalpy of mixing), kJ/mol",...,mol2_desc_190,mol2_desc_191,mol2_desc_192,mol2_desc_193,mol2_desc_194,mol2_desc_195,mol2_desc_196,mol2_desc_197,mol2_desc_198,mol2_desc_199
16575,0.000047,618.20,30000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.156089
102654,0.000761,393.20,60000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.192005
23907,781.400000,283.15,5000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.273923
105239,766.000000,337.40,11370.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.238586
16593,533.100000,523.20,20000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.156089


In [6]:
train_xgboost(X_train, y_train, X_val, y_val)

Test MSE: 0.0467
Test R^2 Score: 0.4964


In [23]:
def cross_validation(model,
                     X: pd.DataFrame,
                     y: pd.Series,
                     n_fold: int = 5,
                     criteria: str = 'r2',
                     eval: bool = False,
                     return_cv_score: bool = False) -> float|None:
    """Helper function to perform cross-validation for the given model and dataset."""
    X = hf.convert_to_numeric(X)
    cv_results = cross_validate(model, X, y, cv=n_fold, scoring=criteria, return_train_score=True)
    if not eval:
        print(pd.DataFrame(cv_results, index=range(1, n_fold+1)))
    return cv_results['test_score'].mean() if return_cv_score else None

In [27]:
cross_validation(XGBRegressor(n_jobs = -1, n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42), X_train, y_train, return_cv_score=True)

   fit_time  score_time  test_score  train_score
1  1.572949    0.016552    0.518173     0.534178
2  0.857855    0.019399    0.501485     0.512400
3  0.719351    0.015533    0.518229     0.527121
4  0.871197    0.017413    0.503971     0.522606
5  0.810091    0.017720    0.507258     0.524124


np.float64(0.5098230802376078)

### 3. Cross-validation and Hyperparameter Tuning ###

In [28]:
import optuna

def objective(trial)-> float:
    # define the hyperparameters range to search
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'random_state': 42, # Can tune this as well if needed
        'n_jobs': -1
    }

    model = XGBRegressor(**params)

    cv_score = cross_validation(model, X_train, y_train, n_fold=5, criteria='r2', eval=True, return_cv_score=True)

    return cv_score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)
print("Best trial:", study.best_trial.number)
print("Best R2:", study.best_value)
print("Best params:", study.best_params)

[I 2026-02-26 22:27:47,331] A new study created in memory with name: no-name-8a0757fe-c3bd-49d6-bb6e-309ca5bd887a
Best trial: 0. Best value: 0.782565:   1%|          | 1/100 [00:09<15:22,  9.31s/it]

[I 2026-02-26 22:27:56,646] Trial 0 finished with value: 0.7825654369952557 and parameters: {'n_estimators': 114, 'learning_rate': 0.18001205670923145, 'max_depth': 10}. Best is trial 0 with value: 0.7825654369952557.


Best trial: 0. Best value: 0.782565:   2%|▏         | 2/100 [00:14<11:00,  6.74s/it]

[I 2026-02-26 22:28:01,582] Trial 1 finished with value: 0.3601555021915866 and parameters: {'n_estimators': 196, 'learning_rate': 0.06213430372800021, 'max_depth': 4}. Best is trial 0 with value: 0.7825654369952557.


Best trial: 0. Best value: 0.782565:   3%|▎         | 3/100 [00:17<08:26,  5.22s/it]

...

[I 2026-02-26 22:44:58,688] Trial 97 finished with value: 0.7917458757578362 and parameters: {'n_estimators': 182, 'learning_rate': 0.18680339556134362, 'max_depth': 9}. Best is trial 72 with value: 0.8038429413087391.


Best trial: 72. Best value: 0.803843:  99%|█████████▉| 99/100 [17:25<00:13, 13.66s/it]

[I 2026-02-26 22:45:12,973] Trial 98 finished with value: 0.8029879938334918 and parameters: {'n_estimators': 196, 'learning_rate': 0.20857759779177942, 'max_depth': 10}. Best is trial 72 with value: 0.8038429413087391.


Best trial: 72. Best value: 0.803843: 100%|██████████| 100/100 [17:40<00:00, 10.60s/it]

[I 2026-02-26 22:45:27,601] Trial 99 finished with value: 0.8030558895803107 and parameters: {'n_estimators': 197, 'learning_rate': 0.19547756424183627, 'max_depth': 10}. Best is trial 72 with value: 0.8038429413087391.
Best trial: 72
Best R2: 0.8038429413087391
Best params: {'n_estimators': 190, 'learning_rate': 0.23014808660801428, 'max_depth': 10}
